 Задание

Ваша задача — реализовать класс Account, который моделирует поведение банковского счёта. Этот класс должен не только выполнять базовые операции, но и вести детальный учёт всех действий, а также предоставлять аналитику по истории операций.

Этап 1. Реализация базового класса Account
Класс должен быть инициализирован с параметрами:

account_holder (str): имя владельца счёта;

balance (float, по умолчанию 0): начальный баланс счёта, не может быть отрицательным.

Атрибуты:

holder: хранит имя владельца;

_balance: приватный атрибут для хранения текущего баланса;

operations_history: список или другая структура для хранения истории операций.

Важно: каждая операция должна храниться не просто как число, а как структурированная информация, например, словарь или кортеж. Минимальный набор данных для операции: тип операции ('deposit' или 'withdraw'), сумма, дата и время операции, текущий баланс после операции, статус ('success' или 'fail').

 

Этап 2. Реалирзация методов

__init__(self, account_holder, balance=0): конструктор;

deposit(self, amount): метод для пополнения счёта:

принимает сумму (должна быть положительной);

в случае успеха обновляет баланс и добавляет запись в историю операций.

withdraw(self, amount): метод для снятия средств:

принимает сумму (должна быть положительной);

проверяет, достаточно ли средств на счёте, если нет — операция не проходит, но ее попытка с статусом 'fail' все равно фиксируется в истории;

в случае успеха обновляет баланс и добавляет запись.

get_balance(self): метод, который возвращает текущий баланс.

get_history(self): метод, который возвращает историю операций.

 

Важно: продумайте, в каком формате его вернуть. Для работы с датой и временем используйте модуль datetime. Получить текущее время можно с помощью datetime.now().

 

Этап 3. Реализация наследования.

Создайте класс CreditAccount(Account), который наследует всю функциональность класса Account и добавляет новую.

Особенности кредитного счёта:

При инициализации принимает дополнительный параметр credit_limit (кредитный лимит). Баланс такого счета может быть отрицательным, но не ниже значения -credit_limit.

По запросу показывает, сколько кредитных средств еще доступно (текущий баланс + кредитный лимит).

В историю операций добавляется информацию о том, были ли использованы кредитные средства в данной операции.


In [1]:
# Импортируем модуль datetime для работы с датой и временем
from datetime import datetime


class Account:
    """Базовый класс банковского счёта."""

    # Конструктор класса: инициализация нового счёта
    def __init__(self, account_holder: str, balance: float = 0):
        # Проверка: начальный баланс не может быть отрицательным
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")
        # Сохраняем имя владельца счёта
        self.holder = account_holder
        # Приватный атрибут баланса (защита от прямого изменения извне)
        self._balance = balance
        # Список для хранения истории всех операций (словари)
        self.operations_history: list[dict] = []

    # Метод пополнения счёта
    def deposit(self, amount: float) -> None:
        # Проверка: сумма пополнения должна быть строго больше нуля
        if amount <= 0:
            raise ValueError("Сумма пополнения должна быть положительной")
        # Увеличиваем баланс на указанную сумму
        self._balance += amount
        # Добавляем запись об успешной операции в историю
        self.operations_history.append({
            'type': 'deposit',                    # Тип операции — пополнение
            'amount': amount,                     # Сумма операции
            'date': datetime.now(),               # Текущая дата и время
            'balance_after': self._balance,       # Баланс после операции
            'status': 'success',                  # Статус — успех
        })

    # Метод снятия средств со счёта
    def withdraw(self, amount: float) -> bool:
        # Проверка: сумма снятия должна быть положительной
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной")
        # Проверка: достаточно ли средств на счёте
        if amount > self._balance:
            # Если средств не хватает — записываем неудачную попытку в историю
            self.operations_history.append({
                'type': 'withdraw',
                'amount': amount,
                'date': datetime.now(),
                'balance_after': self._balance,   # Баланс не изменился
                'status': 'fail',                 # Статус — неудача
            })
            return False  # Возвращаем False, сигнализируя о неудаче
        # Если средств достаточно — уменьшаем баланс
        self._balance -= amount
        # Записываем успешную операцию в историю
        self.operations_history.append({
            'type': 'withdraw',
            'amount': amount,
            'date': datetime.now(),
            'balance_after': self._balance,
            'status': 'success',
        })
        return True  # Возвращаем True — операция успешна

    # Геттер для получения текущего баланса
    def get_balance(self) -> float:
        return self._balance

    # Геттер для получения полной истории операций
    def get_history(self) -> list[dict]:
        return self.operations_history

    # Метод красивого вывода истории операций в консоль
    def print_history(self) -> None:
        """Красивый вывод истории операций."""
        # Задаём ширину каждой колонки для выравнивания таблицы
        w_date, w_type, w_sum, w_status, w_bal = 20, 12, 10, 30, 10
        sep = "  "  # Разделитель между колонками
        # Вычисляем общую ширину таблицы
        width = w_date + w_type + w_sum + w_status + w_bal + len(sep) * 4

        # Печатаем "шапку" выписки
        print("\n" + "=" * width)
        print(f"{'ВЫПИСКА ПО СЧЁТУ':^{width}}")
        print(f"{'Владелец: ' + self.holder:^{width}}")
        print(f"{'Текущий баланс: ' + str(self._balance) + ' ₽':^{width}}")
        print("=" * width)

        # Если операций не было — выводим соответствующее сообщение
        if not self.operations_history:
            print("Операции отсутствуют")
            print("=" * width + "\n")
            return

        # Печатаем заголовки колонок с выравниванием
        print(f"{'Дата':<{w_date}}{sep}"
              f"{'Тип':<{w_type}}{sep}"
              f"{'Сумма':>{w_sum}}{sep}"
              f"{'Статус':<{w_status}}{sep}"
              f"{'Баланс':>{w_bal}}")
        print("-" * width)  # Разделительная линия

        # Проходим по всем записям в истории и печатаем их
        for record in self.operations_history:
            # Форматируем дату в читаемый вид
            date_str = record['date'].strftime('%Y-%m-%d %H:%M:%S')
            # Определяем текстовое название операции
            operation = "Пополнение" if record['type'] == 'deposit' else "Снятие"
            # Проверяем статус операции для иконки
            is_success = record['status'] == 'success'
            status_icon = "✓" if is_success else "✗"
            status_text = f"{status_icon} {record['status']}"

            # Печатаем строку с данными операции
            print(f"{date_str:<{w_date}}{sep}"
                  f"{operation:<{w_type}}{sep}"
                  f"{record['amount']:>{w_sum}.2f}{sep}"
                  f"{status_text:<{w_status}}{sep}"
                  f"{record['balance_after']:>{w_bal}.2f}")

        # Печатаем итоги: разделитель, общее число операций, закрывающая линия
        print("-" * width)
        print(f"Всего операций: {len(self.operations_history)}")
        print("=" * width + "\n")


class CreditAccount(Account):
    """Кредитный счёт: баланс может уходить в минус до -credit_limit."""

    # Конструктор кредитного счёта с дополнительным параметром — кредитным лимитом
    def __init__(self, account_holder: str, balance: float = 0,
                 credit_limit: float = 0):
        # Вызываем конструктор родительского класса
        super().__init__(account_holder, balance)
        # Сохраняем кредитный лимит
        self.credit_limit = credit_limit

    # Переопределяем метод снятия — теперь можно уходить в минус в пределах лимита
    def withdraw(self, amount: float) -> bool:
        if amount <= 0:
            raise ValueError("Сумма снятия должна быть положительной")
        # Проверяем лимит: баланс + кредитный лимит
        if amount > self._balance + self.credit_limit:
            # Если сумма превышает доступные средства — записываем неудачу
            self.operations_history.append({
                'type': 'withdraw',
                'amount': amount,
                'date': datetime.now(),
                'balance_after': self._balance,
                'status': 'fail',
                'credit_used': False,  # Кредит не использован
            })
            return False

        # Уменьшаем баланс (он может стать отрицательным)
        self._balance -= amount
        # Кредит использован, если баланс стал отрицательным
        credit_used = self._balance < 0

        # Записываем успешную операцию с пометкой об использовании кредита
        self.operations_history.append({
            'type': 'withdraw',
            'amount': amount,
            'date': datetime.now(),
            'balance_after': self._balance,
            'status': 'success',
            'credit_used': credit_used,
        })
        return True

    # Метод для расчёта оставшегося доступного кредита
    def get_available_credit(self) -> float:
        """Сколько кредитных средств ещё доступно."""
        # Доступный кредит = текущий баланс (может быть отрицательным) + лимит
        return self._balance + self.credit_limit

    # Переопределяем вывод истории — добавляем колонку "Кредит"
    def print_history(self) -> None:
        """Вывод истории с дополнительной колонкой 'Кредит использован'."""
        # Ширина колонок (добавили w_credit для новой колонки)
        w_date, w_type, w_sum, w_status, w_bal, w_credit = 20, 12, 10, 20, 10, 12
        sep = "  "
        # Общая ширина таблицы с учётом новой колонки
        width = (w_date + w_type + w_sum + w_status + w_bal + w_credit
                 + len(sep) * 5)

        # Печатаем шапку выписки для кредитного счёта
        print("\n" + "=" * width)
        print(f"{'ВЫПИСКА ПО КРЕДИТНОМУ СЧЁТУ':^{width}}")
        print(f"{'Владелец: ' + self.holder:^{width}}")
        print(f"{'Кредитный лимит: ' + str(self.credit_limit) + ' ₽':^{width}}")
        print(f"{'Текущий баланс: ' + str(self._balance) + ' ₽':^{width}}")
        print(f"{'Доступно кредита: ' + str(self.get_available_credit()) + ' ₽':^{width}}")
        print("=" * width)

        # Если операций нет — выводим сообщение
        if not self.operations_history:
            print("Операции отсутствуют")
            print("=" * width + "\n")
            return

        # Печатаем заголовки колонок (включая "Кредит")
        print(f"{'Дата':<{w_date}}{sep}"
              f"{'Тип':<{w_type}}{sep}"
              f"{'Сумма':>{w_sum}}{sep}"
              f"{'Статус':<{w_status}}{sep}"
              f"{'Баланс':>{w_bal}}{sep}"
              f"{'Кредит':>{w_credit}}")
        print("-" * width)

        # Проходим по истории и печатаем каждую операцию
        for record in self.operations_history:
            date_str = record['date'].strftime('%Y-%m-%d %H:%M:%S')
            operation = "Пополнение" if record['type'] == 'deposit' else "Снятие"
            is_success = record['status'] == 'success'
            status_icon = "✓" if is_success else "✗"
            status_text = f"{status_icon} {record['status']}"
            # Определяем, использовался ли кредит в этой операции
            credit_info = "Да" if record.get('credit_used') else "Нет"

            # Печатаем строку с данными операции
            print(f"{date_str:<{w_date}}{sep}"
                  f"{operation:<{w_type}}{sep}"
                  f"{record['amount']:>{w_sum}.2f}{sep}"
                  f"{status_text:<{w_status}}{sep}"
                  f"{record['balance_after']:>{w_bal}.2f}{sep}"
                  f"{credit_info:>{w_credit}}")

        # Итоги: разделитель, количество операций, закрывающая линия
        print("-" * width)
        print(f"Всего операций: {len(self.operations_history)}")
        print("=" * width + "\n")


# ==============================
# Тестирование
# ==============================
# Блок выполняется только при прямом запуске скрипта (не при импорте)
if __name__ == "__main__":
    # --- ТЕСТ 1: Обычный счёт ---
    print("=" * 50)
    print("ТЕСТ 1: Обычный счёт")
    print("=" * 50)
    # Создаём обычный счёт с начальным балансом 1000
    user = Account("Пупкин", balance=1000)
    # Пополняем счёт на 500
    user.deposit(500)
    # Снимаем 200 — операция должна пройти успешно
    user.withdraw(200)
    # Пытаемся снять 2000 — операция должна провалиться (недостаточно средств)
    user.withdraw(2000)  # fail — недостаточно средств
    # Выводим текущий баланс
    print(f"Баланс: {user.get_balance()} ₽")
    # Печатаем полную историю операций
    user.print_history()
    

    # --- ТЕСТ 2: Кредитный счёт ---
    print("=" * 50)
    print("ТЕСТ 2: Кредитный счёт")
    print("=" * 50)
    # Создаём кредитный счёт: баланс 1000, лимит 5000
    credit_user = CreditAccount("Залупкин", balance=1000, credit_limit=5000)
    # Пополняем на 500 (баланс станет 1500)
    credit_user.deposit(500)
    # Снимаем 2000 — успешно, без использования кредита (1500 >= 2000? нет, 1500-2000=-500, кредит использован)
    # На самом деле: баланс 1500, снимаем 2000 — уйдём в минус на 500, кредит будет использован
    credit_user.withdraw(2000)  # success, с использованием кредита
    # Снимаем 3000 — баланс -500, снимаем ещё 3000, станет -3500, лимит 5000 — проходит
    credit_user.withdraw(3000)  # success, с использованием кредита
    # Пытаемся снять 2000 — баланс -3500, лимит 5000, доступно 1500 — не проходит
    credit_user.withdraw(2000)  # fail — превышает лимит
    # Выводим итоговый баланс
    print(f"Баланс: {credit_user.get_balance()} ₽")
    # Выводим доступный остаток кредита
    print(f"Доступно кредита: {credit_user.get_available_credit()} ₽")
    # Печатаем полную историю операций кредитного счёта
    credit_user.print_history()

ТЕСТ 1: Обычный счёт
Баланс: 1300 ₽

                                     ВЫПИСКА ПО СЧЁТУ                                     
                                     Владелец: Пупкин                                     
                                  Текущий баланс: 1300 ₽                                  
Дата                  Тип                Сумма  Статус                              Баланс
------------------------------------------------------------------------------------------
2026-11-08 17:21:24   Пополнение        500.00  ✓ success                          1500.00
2026-11-08 17:21:24   Снятие            200.00  ✓ success                          1300.00
2026-11-08 17:21:24   Снятие           2000.00  ✗ fail                             1300.00
------------------------------------------------------------------------------------------
Всего операций: 3

ТЕСТ 2: Кредитный счёт
Баланс: -3500 ₽
Доступно кредита: 1500 ₽

                                 ВЫПИСКА ПО КРЕДИТНОМУ СЧЁТУ